# Homework #3 - Financial Transactions

**Course:** Big Data Analysis

**Student:** SIMONA GULISANO

---

This project develops a dimensional model for financial transactions, performs ETL and data quality checks, answers analytical questions, and supports an interactive Streamlit dashboard.

# 1. Data Warehouse Modeling: Star Schema

## 1.1 Define the Business Process and the Fact Grain

The business process analyzed in this project is the execution of stock market transactions during the year 2024. The available datasets contain information about financial transactions, traded stocks, companies, and the geographical location associated with each company.

The objective of the dimensional model is to support the analysis of trading activity from multiple perspectives, including time, geography, stock characteristics, and transaction type. By organizing the data according to a star schema structure, the model enables efficient analytical queries and facilitates the development of interactive reporting solutions.

### Fact Grain

Defining the grain of the fact table is a fundamental step in dimensional modeling, as it determines the level of detail represented by each record.

The grain of the fact table is defined as:

> One row in Fact_Transactions represents one transaction from the account statement dataset.

Each transaction contains:

* Transaction ID
* Transaction date
* Transaction type (BUY or SELL)
* Traded stock symbol
* Number of traded units

This level of granularity ensures that all analyses are performed using the most detailed transactional information available in the source data.


## 1.2 Identify Fact and Dimensions

To support multidimensional analysis, the data warehouse is modeled according to a star schema structure composed of one fact table and four dimension tables.

The fact table stores the business events to be analyzed, while the dimension tables provide the descriptive context required to interpret and aggregate transactional data. This separation between quantitative measures and descriptive attributes improves both analytical flexibility and query performance.

### Fact Table

Fact_Transactions

The fact table contains one record for each financial transaction and stores the quantitative measure used for analysis.

Measure:

* Unit

### Dimension Tables

* Dim_Time
* Dim_Geography
* Dim_Symbol
* Dim_TransactionType

These dimensions allow transactions to be analyzed across different perspectives. The Time dimension supports temporal analysis, the Geography dimension enables geographical comparisons, the Symbol dimension provides information about companies and market sectors, and the TransactionType dimension distinguishes between BUY and SELL operations.


## 1.3 Define Dimension Hierarchies

Dimension hierarchies play a fundamental role in a dimensional model, as they allow data to be analyzed at different levels of granularity. Hierarchies support aggregation operations and enable users to navigate data through drill-down and roll-up analyses.

In the proposed star schema, hierarchies are defined for the Time, Geography, and Symbol dimensions. These hierarchies provide a structured view of the data and facilitate the generation of analytical reports from different business perspectives.

### Dim_Time

The Time dimension supports temporal analysis of financial transactions. Users can analyze transaction activity at a detailed daily level or aggregate results at higher levels of the hierarchy.

Hierarchy:

Day → Month → Quarter → Year

### Dim_Geography

The Geography dimension enables the analysis of transactions according to the geographical location associated with the traded company. Following the structure required for this project, the hierarchy is defined as:

Hierarchy:

Country → Sub-region → Region

This hierarchy allows transactions to be grouped and compared across broader geographical areas while preserving the country-level detail.

### Dim_Symbol

The Symbol dimension contains information about the traded companies and their market classification. The hierarchy is designed to support analysis from individual stocks up to broader business categories.

Hierarchy:

Sector → Industry → Company → Symbol

This structure allows users to analyze transaction activity at different organizational levels, from specific companies to entire sectors of the market.

### Dim_TransactionType

The TransactionType dimension contains only two categories, BUY and SELL. Since no additional aggregation levels are available, a hierarchy is not defined for this dimension.


## 1.4 Design the Star Schema

Based on the identified business requirements, a star schema was designed to support the analysis of financial transactions. The model consists of one central fact table connected to four dimension tables through surrogate keys.

The fact table stores the transactional events, while the dimensions provide the descriptive context required for analytical processing. This structure simplifies query execution and enables efficient aggregation across different business perspectives.

### Fact Table

Fact_Transactions

The fact table contains one record for each financial transaction and references all dimensions through foreign keys.

Foreign Keys:

* TimeKey
* GeographyKey
* SymbolKey
* TransactionTypeKey

Measure:

* Unit

### Dimension Tables

Dim_Time

The Time dimension contains the temporal attributes associated with each transaction and supports time-based analyses.

Primary Key:

* TimeKey

Attributes:

* Date
* Day
* Month
* Quarter
* Year
* Weekday

---

Dim_Geography

The Geography dimension contains the geographical information associated with the traded company.

Primary Key:

* GeographyKey

Attributes:

* Country
* SubRegion
* Region

---

Dim_Symbol

The Symbol dimension contains descriptive information about traded stocks and companies.

Primary Key:

* SymbolKey

Attributes:

* Symbol
* CompanyName
* Industry
* Sector
* Country

---

Dim_TransactionType

The TransactionType dimension classifies transactions according to their type.

Primary Key:

* TransactionTypeKey

Attributes:

* TransactionType

Possible values:

* BUY
* SELL

### Star Schema Structure

The final dimensional model follows a star schema architecture where Fact_Transactions is positioned at the center of the model and connected to all dimensions through foreign key relationships.

The corresponding star schema diagram is provided separately as part of the project deliverables.


### Star Schema Diagram

The following diagram represents the final star schema designed for the financial transactions data warehouse.

![Star Schema](StarSchema_FinancialTransactions.png)

# 2. Data Transformation and Analysis

## 2.1 Load and Clean the Data

Before building the dimensional model, the source datasets must be loaded, cleaned, and validated. Data quality checks are performed to identify missing values, remove unnecessary attributes, and verify the consistency of relationships across datasets.

The ETL process includes:

- Loading the source datasets.
- Removing unused attributes.
- Identifying and handling missing values.
- Validating stock symbols referenced in transactions.
- Validating country mappings between datasets.
- Preparing the data required for the dimensional model.

### Import Required Libraries

The following libraries are used for data manipulation and analysis.


In [95]:
# Import required libraries

import pandas as pd
import numpy as np

### Load Source Datasets

The project uses three datasets:

- country.csv: geographical information and regional classifications.
- symbols.csv: stock symbols and company information.
- account-statement.csv: financial transaction records.

In [96]:
# Load source datasets

country = pd.read_csv("Datasets/country.csv")

symbols = pd.read_csv(
    "Datasets/symbols.csv",
    sep=";"
)

transactions = pd.read_csv(
    "Datasets/account-statement-1-1-2024-12-31-2024.csv",
    sep=";"
)

In [97]:
# Display dataset dimensions

print("Country:", country.shape)
print("Symbols:", symbols.shape)
print("Transactions:", transactions.shape)

Country: (249, 11)
Symbols: (3194, 5)
Transactions: (2745, 6)


### Data Quality Assessment

An initial assessment was performed to evaluate the structure and quality of the datasets before applying any transformations.

In [98]:
# Display dataset columns

print("COUNTRY")
print(country.columns)

print("\nSYMBOLS")
print(symbols.columns)

print("\nTRANSACTIONS")
print(transactions.columns)

COUNTRY
Index(['name', 'alpha-2', 'alpha-3', 'country-code', 'iso_3166-2', 'region',
       'sub-region', 'intermediate-region', 'region-code', 'sub-region-code',
       'intermediate-region-code'],
      dtype='str')

SYMBOLS
Index(['symbol', 'company_name', 'sector', 'industry', 'country'], dtype='str')

TRANSACTIONS
Index(['IDTransaction', 'Date', 'TransactionType', 'Symbol', 'Unit',
       'Unnamed: 5'],
      dtype='str')


### Identify Data Quality Issues

A preliminary inspection of the datasets highlighted a number of data quality issues that required further investigation.

The transaction dataset contains an additional column named `Unnamed: 5`, which does not provide any useful information for the analysis.

Furthermore, missing values were detected in both the country and transaction datasets. Therefore, a detailed assessment was performed before proceeding with the dimensional modeling process.

In [99]:
# Check missing values

print("COUNTRY")
print(country.isnull().sum())

print("\nSYMBOLS")
print(symbols.isnull().sum())

print("\nTRANSACTIONS")
print(transactions.isnull().sum())

COUNTRY
name                          0
alpha-2                       1
alpha-3                       0
country-code                  0
iso_3166-2                    0
region                        2
sub-region                    2
intermediate-region         144
region-code                   2
sub-region-code               2
intermediate-region-code    144
dtype: int64

SYMBOLS
symbol          0
company_name    0
sector          0
industry        0
country         0
dtype: int64

TRANSACTIONS
IDTransaction       464
Date                464
TransactionType     464
Symbol              464
Unit                464
Unnamed: 5         2745
dtype: int64


### Remove Unused Attributes

The transaction dataset contains an additional attribute (`Unnamed: 5`) that is completely empty and does not contribute to the analysis. Therefore, it is removed from the dataset.

In [100]:
# Remove unnecessary column

transactions = transactions.drop(columns=["Unnamed: 5"])

transactions.columns


Index(['IDTransaction', 'Date', 'TransactionType', 'Symbol', 'Unit'], dtype='str')

### Remove Empty Records

The missing values detected in the transaction dataset correspond to completely empty rows located at the end of the file.

Since these rows do not contain any transaction information, they are removed from the dataset.

In [101]:
# Remove fully empty rows

transactions = transactions.dropna(how="all")

print(transactions.shape)

print("\nMissing values after cleaning:")
print(transactions.isnull().sum())

(2281, 5)

Missing values after cleaning:
IDTransaction      0
Date               0
TransactionType    0
Symbol             0
Unit               0
dtype: int64


### Data Consistency Validation

After cleaning the datasets, additional validation checks were performed to verify the consistency of relationships across the different data sources.

In particular, two validation procedures were conducted:

- Verification that all transaction symbols could be mapped to the symbol reference dataset.
- Verification that all company countries could be mapped to the country reference dataset.

In [102]:
# Validate transaction symbols

missing_symbols = set(
    transactions["Symbol"]
) - set(
    symbols["symbol"]
)

print("Missing symbols:", sorted(missing_symbols))
print("Number of missing symbols:", len(missing_symbols))

Missing symbols: ['AGO.l', 'ARCH', 'AZM', 'CCAP', 'CSIQ', 'FNC', 'HTGC', 'IBE', 'MFG', 'MONC', 'OBDC', 'RCMT', 'RIGZU', 'SAP', 'TKC', 'UCG', 'VWS', 'WF']
Number of missing symbols: 18


#### Symbol Validation Results

The validation process identified a small number of transaction symbols that could not be matched to the symbol reference dataset.

A total of 18 symbols were found in the transaction dataset but were not present in the symbols dataset. These records were retained for analysis but could not be associated with company-level descriptive information.

In [103]:
# Validate country mappings

missing_countries = set(
    symbols["country"]
) - set(
    country["name"]
)

print("Missing countries:", missing_countries)
print("Number of missing countries:", len(missing_countries))

Missing countries: {'Turkey', 'Taiwan'}
Number of missing countries: 2


#### Country Validation Results

The validation process identified two country names that were not directly present in the country reference dataset:

- Turkey
- Taiwan

A manual inspection revealed that these countries were represented using different naming conventions in the country dataset.

In [104]:
# Inspect Taiwan entry

country[country["name"].str.contains("Taiwan", case=False, na=False)]

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
217,"Taiwan, Province of China",TW,TWN,158,ISO 3166-2:TW,NaN,NaN,NaN,NaN,NaN,NaN


In [105]:
# Inspect Türkiye entry

country[country["name"].str.contains("Tur", case=False, na=False)]

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
228,Turkmenistan,TM,TKM,795,ISO 3166-2:TM,Asia,Central Asia,NaN,142.0,143.0,NaN
229,Turks and Caicos Islands,TC,TCA,796,ISO 3166-2:TC,Americas,Latin America and the Caribbean,Caribbean,19.0,419.0,29.0


In [106]:
country[country["alpha-2"] == "TR"]

,name,alpha-2,alpha-3,country-code,iso_3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
227,Türkiye,TR,TUR,792,ISO 3166-2:TR,Asia,Western Asia,NaN,142.0,145.0,NaN


#### Country Validation Results

The validation process identified two country names present in the symbols dataset but not directly found in the country reference dataset:

- Turkey
- Taiwan

To investigate these discrepancies, a manual inspection of the country reference dataset was performed.

The inspection revealed that both countries were already present in the reference dataset under different official names:

- Turkey was recorded as **Türkiye** (ISO code: TR).
- Taiwan was recorded as **Taiwan, Province of China** (ISO code: TW).

Therefore, the mismatch was caused by different naming conventions adopted across the source datasets rather than by missing country records.

In [107]:
# Standardize country names

symbols["country"] = symbols["country"].replace({
    "Turkey": "Türkiye",
    "Taiwan": "Taiwan, Province of China"
})

### Create Analytical Dataset

To support the analytical queries, the transaction dataset was integrated with company and geographical information.

The resulting analytical dataset combines:

- Transaction information
- Company information
- Industry and sector classifications
- Country and geographical attributes

This integrated dataset will be used to answer the analytical questions required by the assignment.

In [108]:
# Convert date column

transactions["Date"] = pd.to_datetime(
    transactions["Date"],
    dayfirst=True
)

transactions["Date"].head()

0   2024-01-11 10:44:03
1   2024-01-24 08:07:24
2   2024-01-10 11:00:08
3   2024-01-16 08:14:21
4   2024-01-16 14:34:12
Name: Date, dtype: datetime64[us]

In [109]:
# Merge transactions with symbols

analysis_df = transactions.merge(
    symbols,
    left_on="Symbol",
    right_on="symbol",
    how="left"
)

analysis_df.shape

(2281, 10)

In [110]:
# Merge with geography information

analysis_df = analysis_df.merge(
    country,
    left_on="country",
    right_on="name",
    how="left"
)

analysis_df.shape

(2281, 21)

In [111]:
# Create time attributes

analysis_df["Year"] = analysis_df["Date"].dt.year
analysis_df["Quarter"] = analysis_df["Date"].dt.quarter
analysis_df["Month"] = analysis_df["Date"].dt.month
analysis_df["Weekday"] = analysis_df["Date"].dt.day_name()

In [112]:
analysis_df.head()

,IDTransaction,Date,TransactionType,Symbol,Unit,symbol,company_name,sector,industry,country,...,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code,Year,Quarter,Month,Weekday
0,2.769834e+09,2024-01-11 10:44:03,BUY,BAP,1605.0,BAP,Credicorp Ltd.,Financial Services,Banks - Regional,Peru,...,Americas,Latin America and the Caribbean,South America,19.0,419.0,5.0,2024,1,1,Thursday
1,2.767325e+09,2024-01-24 08:07:24,SELL,BAP,1605.0,BAP,Credicorp Ltd.,Financial Services,Banks - Regional,Peru,...,Americas,Latin America and the Caribbean,South America,19.0,419.0,5.0,2024,1,1,Wednesday
2,2.815474e+09,2024-01-10 11:00:08,SELL,BAP,914.0,BAP,Credicorp Ltd.,Financial Services,Banks - Regional,Peru,...,Americas,Latin America and the Caribbean,South America,19.0,419.0,5.0,2024,1,1,Wednesday
3,2.622244e+09,2024-01-16 08:14:21,BUY,ACGL,646.0,ACGL,Arch Capital Group Ltd.,Financial Services,Insurance - Diversified,Bermuda,...,Americas,Northern America,NaN,19.0,21.0,NaN,2024,1,1,Tuesday
4,2.629871e+09,2024-01-16 14:34:12,SELL,ALVO,646.0,ALVO,Alvotech,Healthcare,Drug Manufacturers - Specialty & Generic,Luxembourg,...,Europe,Western Europe,NaN,150.0,155.0,NaN,2024,1,1,Tuesday


In [113]:
analysis_df.columns

Index(['IDTransaction', 'Date', 'TransactionType', 'Symbol', 'Unit', 'symbol',
       'company_name', 'sector', 'industry', 'country', 'name', 'alpha-2',
       'alpha-3', 'country-code', 'iso_3166-2', 'region', 'sub-region',
       'intermediate-region', 'region-code', 'sub-region-code',
       'intermediate-region-code', 'Year', 'Quarter', 'Month', 'Weekday'],
      dtype='str')

In [114]:
# Keep only rows with complete analytical information

analysis_df = analysis_df.dropna(
    subset=["sector", "industry", "country", "region"]
)

analysis_df.shape

(1975, 25)

## 2.2 Analytical Questions

After completing the data cleaning and validation process, the integrated analytical dataset is used to answer a selection of business-oriented analytical questions.

The following analyses explore transaction patterns across different dimensions of the star schema, including time, geography, sector, industry, and transaction type.

### Question 1

#### What are the top 5 sectors by number of SELL transactions in US during 2024?

This analysis identifies the sectors generating the highest number of SELL transactions for companies located in the United States during 2024.

In [115]:
q1 = (
    analysis_df[
        (analysis_df["TransactionType"] == "SELL") &
        (analysis_df["country"] == "United States of America")
    ]
    .groupby("sector")
    .size()
    .sort_values(ascending=False)
    .head(5)
)

q1

sector
Technology                158
Communication Services     58
Financial Services         55
Healthcare                 50
Consumer Cyclical          48
dtype: int64

### Question 2

#### What are the top 5 industries by number of BUY transactions in Q4 of 2024?

The objective is to identify the industries receiving the highest number of BUY transactions during the fourth quarter of 2024.

In [116]:
q2 = (
    analysis_df[
        (analysis_df["TransactionType"] == "BUY") &
        (analysis_df["Quarter"] == 4)
    ]
    .groupby("industry")
    .size()
    .sort_values(ascending=False)
    .head(5)
)

q2

industry
Internet Content & Information    15
Semiconductors                    13
Software - Infrastructure         10
Internet Retail                    8
Diagnostics & Research             7
dtype: int64

### Question 3

#### What are the top 10 industry-region pairs by total number of transactions in 2024?

This analysis identifies the combinations of industries and geographical regions that generated the highest transaction activity during 2024.

In [117]:
q3 = (
    analysis_df
    .groupby(["industry", "region"])
    .size()
    .reset_index(name="Transactions")
    .sort_values(
        by="Transactions",
        ascending=False
    )
    .head(10)
)

q3

,industry,region,Transactions
59,Semiconductors,Americas,134
64,Software - Infrastructure,Americas,132
38,Internet Content & Information,Americas,119
60,Semiconductors,Europe,100
70,Telecom Services,Americas,81
11,Biotechnology,Europe,66
61,Software - Application,Americas,61
7,Auto Manufacturers,Asia,60
10,Biotechnology,Americas,59
18,Credit Services,Americas,48


### Question 4

#### What are the top 10 countries by number of SELL transactions in 2024?

The analysis highlights the countries associated with the highest volume of SELL transactions.

In [118]:
q4 = (
    analysis_df[
        analysis_df["TransactionType"] == "SELL"
    ]
    .groupby("country")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

q4

country
United States of America                                389
United Kingdom of Great Britain and Northern Ireland    130
China                                                   112
Brazil                                                   69
Netherlands, Kingdom of the                              46
Switzerland                                              37
Ireland                                                  31
Luxembourg                                               27
Canada                                                   22
Germany                                                  18
dtype: int64

### Question 5

#### What are the top 5 regions by total units bought in 2024?

This analysis evaluates the geographical regions with the highest quantity of purchased units during 2024.

In [119]:
q5 = (
    analysis_df[
        analysis_df["TransactionType"] == "BUY"
    ]
    .groupby("region")["Unit"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

q5

region
Americas    37026.0
Europe      22528.0
Asia         9198.0
Name: Unit, dtype: float64

### Question 6

#### What are the top 5 sectors by total units traded (BUY + SELL) in Q3 of 2024?

This analysis identifies the sectors with the highest total number of traded units (BUY and SELL combined) during the third quarter of 2024.

In [120]:
q6 = (
    analysis_df[
        analysis_df["Quarter"] == 3
    ]
    .groupby("sector")["Unit"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

q6

sector
Technology                4240.0
Healthcare                3771.0
Consumer Cyclical         2667.0
Financial Services        2358.0
Communication Services    2116.0
Name: Unit, dtype: float64

### Question 7

#### What are the top 10 symbols by number of transactions (BUY + SELL) in 2024?

This analysis identifies the stock symbols with the highest number of transactions during 2024.

In [121]:
q7 = (
    analysis_df
    .groupby("Symbol")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

q7

Symbol
ARM     100
AMD      97
TIMB     76
GOOG     52
MSFT     49
AMZN     47
ARDX     43
BRFS     42
BLK      42
EH       40
dtype: int64

### Question 8

#### What are the top 5 regions by number of distinct industries traded?

This analysis identifies the geographical regions with the highest diversity of traded industries.

In [122]:
q8 = (
    analysis_df
    .groupby("region")["industry"]
    .nunique()
    .sort_values(ascending=False)
    .head(5)
)

q8

region
Americas    38
Europe      26
Asia         9
Name: industry, dtype: int64